In [ ]:
import numpy as np
import plotly.graph_objects as go

import qubex as qx
from qubex.measurement import StateClassifierGMM

# ex = Experiment(...) と ex.connect() は実行済みとする
ex = globals().get("ex")
if ex is None:
    raise RuntimeError("先に ex = Experiment(...) と ex.connect() を実行してください。")

TARGET = "Q22"
READOUT = ex.experiment_system.resolve_read_label(TARGET)

# find_optimal_readout_frequency() の既定値と同じ 20 MHz 幅・400 kHz 刻み
CENTER_FREQUENCY_GHZ = float(ex.targets[READOUT].frequency)
FREQUENCY_WIDTH_GHZ = 0.020
DF_GHZ = 0.0004

N_SHOTS = 2048
SHOT_INTERVAL_NS = 150 * 1024
INTER_READOUT_DELAY_NS = 128
READOUT_AMPLITUDE = None  # None: 現在の校正値を使用
READOUT_DURATION_NS = None  # None: Experiment の既定値を使用
MIN_ASSIGNMENT_FIDELITY = 0.80
RANDOM_SEED = 7

if TARGET not in ex.qubit_labels:
    raise ValueError(f"{TARGET} is not in ex.qubit_labels: {ex.qubit_labels}")

frequencies_ghz = np.arange(
    CENTER_FREQUENCY_GHZ - FREQUENCY_WIDTH_GHZ / 2,
    CENTER_FREQUENCY_GHZ + FREQUENCY_WIDTH_GHZ / 2,
    DF_GHZ,
)

print(f"target       : {TARGET}")
print(f"readout      : {READOUT}")
print(f"center       : {CENTER_FREQUENCY_GHZ:.6f} GHz")
print(f"sweep points : {len(frequencies_ghz)}")
print(f"range        : {frequencies_ghz[0]:.6f} .. {frequencies_ghz[-1]:.6f} GHz")

In [ ]:
x180_pulse = ex.x180(TARGET)


def make_qnd_schedule(frequency_ghz, prepared_state):
    """Prepare |0> or |1>, then apply two consecutive readout pulses."""
    if prepared_state not in (0, 1):
        raise ValueError("prepared_state must be 0 or 1.")

    schedule = qx.PulseSchedule([TARGET, READOUT])
    with schedule as s:
        if prepared_state == 0:
            # |1> 準備と同じ時間だけ待ち、読み出し開始時刻をそろえる
            s.add(TARGET, qx.Blank(duration=x180_pulse.duration))
        else:
            s.add(TARGET, x180_pulse)
        s.barrier()
        s.add(
            READOUT,
            ex.readout(
                READOUT,
                duration=READOUT_DURATION_NS,
                amplitude=READOUT_AMPLITUDE,
            ),
        )
        s.add(READOUT, qx.Blank(duration=INTER_READOUT_DELAY_NS))
        s.add(
            READOUT,
            ex.readout(
                READOUT,
                duration=READOUT_DURATION_NS,
                amplitude=READOUT_AMPLITUDE,
            ),
        )

    schedule.set_frequency(READOUT, float(frequency_ghz))
    return schedule


# 代表点で readout pulse が2個あることを、測定前に確認
preview_schedule = make_qnd_schedule(CENTER_FREQUENCY_GHZ, prepared_state=1)
preview_measurement_schedule = ex.build_measurement_schedule(
    preview_schedule,
    final_measurement=False,
)
preview_captures = [
    capture
    for capture in preview_measurement_schedule.capture_schedule.captures
    if not capture.is_workaround
]
if len(preview_captures) != 2:
    raise RuntimeError(f"Expected two readout captures, got {len(preview_captures)}.")

print("capture windows:")
for index, capture in enumerate(preview_captures):
    print(
        f"  M{index + 1}: start={capture.start_time:.1f} ns, "
        f"duration={capture.duration:.1f} ns"
    )

In [ ]:
sweep_by_prepared = {}

for prepared_state in (0, 1):
    print(f"sweeping prepared |{prepared_state}> ...")
    sweep_by_prepared[prepared_state] = await ex.run_sweep_measurement(
        schedule=lambda frequency, state=prepared_state: make_qnd_schedule(
            frequency,
            state,
        ),
        sweep_values=frequencies_ghz,
        n_shots=N_SHOTS,
        shot_interval=SHOT_INTERVAL_NS,
        shot_averaging=False,
        time_integration=True,
        state_classification=False,
        final_measurement=False,
        enable_tqdm=True,
    )

output_target = TARGET
if output_target not in sweep_by_prepared[0].data:
    output_target = READOUT

captures_by_prepared = {
    state: [
        np.asarray(capture) for capture in sweep_by_prepared[state].data[output_target]
    ]
    for state in (0, 1)
}

expected_shape = (len(frequencies_ghz), N_SHOTS)
for state in (0, 1):
    captures = captures_by_prepared[state]
    if len(captures) != 2:
        raise RuntimeError(
            f"Expected exactly two captures for |{state}>, got {len(captures)}."
        )
    for capture_index, values in enumerate(captures):
        if values.shape != expected_shape:
            raise RuntimeError(
                f"Unexpected shape for |{state}> capture {capture_index}: "
                f"{values.shape}, expected {expected_shape}."
            )

print("capture shape:", captures_by_prepared[0][0].shape)

In [ ]:
def binomial_standard_error(probability, count):
    """Return the standard error of a binomial probability estimate."""
    if count <= 0 or not np.isfinite(probability):
        return np.nan
    return np.sqrt(probability * (1.0 - probability) / count)


metric_names = (
    "qnd_fidelity",
    "qnd_standard_error",
    "p_m2_0_given_m1_0",
    "p_m2_1_given_m1_1",
    "assignment_fidelity",
    "assignment_0",
    "assignment_1",
    "n_m1_0",
    "n_m1_1",
)
metrics = {
    name: np.full(len(frequencies_ghz), np.nan, dtype=float) for name in metric_names
}
classifiers = [None] * len(frequencies_ghz)
analysis_failures = {}

for index, frequency in enumerate(frequencies_ghz):
    first_0 = np.asarray(captures_by_prepared[0][0][index]).ravel()
    first_1 = np.asarray(captures_by_prepared[1][0][index]).ravel()
    second_0 = np.asarray(captures_by_prepared[0][1][index]).ravel()
    second_1 = np.asarray(captures_by_prepared[1][1][index]).ravel()

    try:
        classifier = StateClassifierGMM.fit(
            {0: first_0, 1: first_1},
            n_init=10,
            random_state=RANDOM_SEED,
        )
        classifiers[index] = classifier

        first_labels_0 = classifier.predict(first_0)
        first_labels_1 = classifier.predict(first_1)
        second_labels_0 = classifier.predict(second_0)
        second_labels_1 = classifier.predict(second_1)

        assignment_0 = np.mean(first_labels_0 == 0)
        assignment_1 = np.mean(first_labels_1 == 1)

        first_labels = np.concatenate([first_labels_0, first_labels_1])
        second_labels = np.concatenate([second_labels_0, second_labels_1])
        first_is_0 = first_labels == 0
        first_is_1 = first_labels == 1

        n_m1_0 = int(np.count_nonzero(first_is_0))
        n_m1_1 = int(np.count_nonzero(first_is_1))
        p_0_to_0 = np.mean(second_labels[first_is_0] == 0) if n_m1_0 > 0 else np.nan
        p_1_to_1 = np.mean(second_labels[first_is_1] == 1) if n_m1_1 > 0 else np.nan

        se_0 = binomial_standard_error(p_0_to_0, n_m1_0)
        se_1 = binomial_standard_error(p_1_to_1, n_m1_1)
        metrics["qnd_fidelity"][index] = 0.5 * (p_0_to_0 + p_1_to_1)
        metrics["qnd_standard_error"][index] = 0.5 * np.sqrt(se_0**2 + se_1**2)
        metrics["p_m2_0_given_m1_0"][index] = p_0_to_0
        metrics["p_m2_1_given_m1_1"][index] = p_1_to_1
        metrics["assignment_fidelity"][index] = 0.5 * (assignment_0 + assignment_1)
        metrics["assignment_0"][index] = assignment_0
        metrics["assignment_1"][index] = assignment_1
        metrics["n_m1_0"][index] = n_m1_0
        metrics["n_m1_1"][index] = n_m1_1
    except (ValueError, FloatingPointError) as error:
        analysis_failures[float(frequency)] = f"{type(error).__name__}: {error}"

valid_mask = (
    np.isfinite(metrics["qnd_fidelity"])
    & np.isfinite(metrics["assignment_fidelity"])
    & (metrics["assignment_fidelity"] >= MIN_ASSIGNMENT_FIDELITY)
)
if not np.any(valid_mask):
    raise RuntimeError(
        "No valid sweep point remains. Expand the sweep range, lower "
        "MIN_ASSIGNMENT_FIDELITY, or inspect the IQ distributions."
    )

optimization_values = np.where(valid_mask, metrics["qnd_fidelity"], np.nan)
optimal_index = int(np.nanargmax(optimization_values))
optimal_frequency_ghz = float(frequencies_ghz[optimal_index])
optimal_qnd_fidelity = float(metrics["qnd_fidelity"][optimal_index])
optimal_qnd_standard_error = float(metrics["qnd_standard_error"][optimal_index])

In [ ]:
print(f"optimal readout frequency : {optimal_frequency_ghz:.7f} GHz")
print(
    "QND fidelity             : "
    f"{optimal_qnd_fidelity:.5f} ± {optimal_qnd_standard_error:.5f} (1σ)"
)
print(f"P(m2=0 | m1=0)          : {metrics['p_m2_0_given_m1_0'][optimal_index]:.5f}")
print(f"P(m2=1 | m1=1)          : {metrics['p_m2_1_given_m1_1'][optimal_index]:.5f}")
print(f"assignment fidelity      : {metrics['assignment_fidelity'][optimal_index]:.5f}")
print(f"valid points              : {np.count_nonzero(valid_mask)} / {len(valid_mask)}")
print(f"GMM fit failures          : {len(analysis_failures)}")

figure = go.Figure()
figure.add_scatter(
    x=frequencies_ghz,
    y=metrics["qnd_fidelity"],
    error_y={
        "type": "data",
        "array": metrics["qnd_standard_error"],
        "visible": True,
    },
    name="QND fidelity",
    mode="lines+markers",
)
figure.add_scatter(
    x=frequencies_ghz,
    y=metrics["assignment_fidelity"],
    name="Assignment fidelity (M1)",
    mode="lines+markers",
)
if np.any(~valid_mask & np.isfinite(metrics["qnd_fidelity"])):
    figure.add_scatter(
        x=frequencies_ghz[~valid_mask],
        y=metrics["qnd_fidelity"][~valid_mask],
        name="Excluded by assignment guard",
        mode="markers",
        marker={"color": "gray", "symbol": "x", "size": 10},
    )
figure.add_hline(
    y=MIN_ASSIGNMENT_FIDELITY,
    line_dash="dot",
    line_color="gray",
    annotation_text="assignment threshold",
)
figure.add_vline(
    x=optimal_frequency_ghz,
    line_width=2,
    line_color="red",
    opacity=0.7,
)
figure.update_layout(
    title=f"QND readout-frequency optimization: {TARGET}",
    xaxis_title="Readout frequency (GHz)",
    yaxis_title="Fidelity",
    yaxis_range=[0, 1.02],
    width=850,
    height=480,
)
figure.show()